In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path(
    r"C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\image\data\processed"
)

manifest = pd.read_csv(data_dir / "clean_image_manifest.csv")
review = pd.read_csv(data_dir / "duplicate_review.csv")

print("Grade distribution:")
print(manifest["Tissue_Grade_Clean"].value_counts(dropna=False))

print("\nCases requiring duplicate review:")
print(
    review[
        review["Status"].str.contains(
            "manual review required", na=False
        )
    ].to_string(index=False)
)

Grade distribution:
Tissue_Grade_Clean
A+     530
A      314
B      165
NaN     15
Name: count, dtype: int64

Cases requiring duplicate review:
          Tissue_ID       Normalized_ID  File_Count  Distinct_Hashes                                           Status
NEB-2024-08-16259 R neb-2024-08-16259 r           4                2 Different image contents; manual review required
NEB-2025-09-17471 R neb-2025-09-17471 r           4                2 Different image contents; manual review required


In [2]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading"
)

dataset_path = (
    project_dir
    / "ml/grading/data/processed/cleaned_grading_dataset.xlsx"
)

manifest_path = (
    project_dir
    / "ml/grading/image/data/processed/clean_image_manifest.csv"
)

dataset = pd.read_excel(dataset_path)
manifest = pd.read_csv(manifest_path)

print("Dataset columns related to donor or tissue:")
print([
    col for col in dataset.columns
    if any(term in col.lower() for term in [
        "donor", "tissue", "serial", "eye", "patient"
    ])
])

print("\nImage manifest columns:")
print(manifest.columns.tolist())

print("\nDataset shape:", dataset.shape)
print("Image manifest shape:", manifest.shape)

Dataset columns related to donor or tissue:
['Tissue_ID', 'Eye', 'Tissue_Grade', 'Overall_Donor_Quality', 'Donor_Age', 'Tissue_Serial', 'Donor_Sex', 'Tissue_Outcome', 'Recipient_Eye', 'Donor_Graft_Size_mm', 'VA_Operative_Eye', 'VA_Fellow_Eye', 'Tissue_Grade_Source', 'Tissue_Grade_Clean', 'Donor_Graft_Size_Clean_mm']

Image manifest columns:
['Tissue_ID', 'Normalized_ID', 'Tissue_Grade_Clean', 'Selected_Image', 'Original_Filename', 'SHA256', 'Source_File_Count', 'Byte_Identical_Copy_Count', 'Has_Grade']

Dataset shape: (1054, 116)
Image manifest shape: (1024, 9)


In [3]:
# Inspect possible donor/tissue identifiers
for col in ["Tissue_ID", "Tissue_Serial", "Donor_Age"]:
    print(f"\n--- {col} ---")
    print("Non-null values:", dataset[col].notna().sum())
    print("Unique values:", dataset[col].nunique())
    print("Missing values:", dataset[col].isna().sum())

print("\nTissue_Serial examples:")
print(dataset["Tissue_Serial"].dropna().head(10).tolist())

print("\nRepeated Tissue_Serial values:")
print(
    dataset.loc[
        dataset["Tissue_Serial"].notna()
        & dataset["Tissue_Serial"].duplicated(keep=False),
        ["Tissue_ID", "Tissue_Serial", "Donor_Age", "Eye"]
    ]
    .sort_values("Tissue_Serial")
    .head(20)
    .to_string(index=False)
)


--- Tissue_ID ---
Non-null values: 1054
Unique values: 1054
Missing values: 0

--- Tissue_Serial ---
Non-null values: 1054
Unique values: 1054
Missing values: 0

--- Donor_Age ---
Non-null values: 1050
Unique values: 57
Missing values: 4

Tissue_Serial examples:
[12346, 12654, 12655, 12656, 12657, 12658, 12659, 12660, 12661, 12662]

Repeated Tissue_Serial values:
Empty DataFrame
Columns: [Tissue_ID, Tissue_Serial, Donor_Age, Eye]
Index: []


In [4]:
import re

def get_tissue_base(tissue_id):
    if pd.isna(tissue_id):
        return None

    value = re.sub(r"\s+", " ", str(tissue_id)).strip()
    value = re.sub(r"^NEB-\s*", "NEB-", value, flags=re.IGNORECASE)

    # Remove a trailing eye indicator (L or R)
    value = re.sub(r"\s+[LR]$", "", value, flags=re.IGNORECASE)

    return value.casefold()


dataset["Tissue_Base_ID"] = dataset["Tissue_ID"].apply(get_tissue_base)

base_counts = dataset.groupby("Tissue_Base_ID").agg(
    Record_Count=("Tissue_ID", "size"),
    Unique_Eyes=("Eye", "nunique"),
    Tissue_IDs=("Tissue_ID", lambda x: list(x))
)

print("Unique base IDs:", len(base_counts))
print("Base IDs with multiple records:", (base_counts["Record_Count"] > 1).sum())

print("\nExamples with multiple records:")
print(
    base_counts[base_counts["Record_Count"] > 1]
    .head(20)
    .to_string()
)

Unique base IDs: 1054
Base IDs with multiple records: 0

Examples with multiple records:
Empty DataFrame
Columns: [Record_Count, Unique_Eyes, Tissue_IDs]
Index: []


In [6]:
print(dataset[
    dataset["Tissue_ID"].astype(str).str.contains(
        "NEB-2024-08-16259", case=False, na=False
    )
][["Tissue_ID", "Tissue_Serial", "Eye", "Tissue_Grade_Clean"]])

               Tissue_ID  Tissue_Serial    Eye Tissue_Grade_Clean
325  NEB-2024-08-16259 R          16259  Right                 A+


In [7]:
import re
import pandas as pd

def extract_donor_base(tissue_id):
    if pd.isna(tissue_id):
        return None

    match = re.match(
        r"^\s*(NEB-\s*\d{4}-\d{2}-\d+)\s+([LR])\s*$",
        str(tissue_id),
        flags=re.IGNORECASE
    )

    if not match:
        return None

    base_id = re.sub(r"\s+", "", match.group(1)).casefold()
    eye = match.group(2).upper()
    return base_id, eye

parsed = dataset["Tissue_ID"].apply(extract_donor_base)

dataset["Donor_Base_ID"] = parsed.apply(
    lambda x: x[0] if x else None
)
dataset["Eye_Suffix"] = parsed.apply(
    lambda x: x[1] if x else None
)

eye_pairs = dataset.groupby("Donor_Base_ID")["Eye_Suffix"].agg(
    lambda eyes: set(eyes.dropna())
)

both_eyes = eye_pairs[eye_pairs.apply(lambda eyes: {"L", "R"}.issubset(eyes))]

print("Records with a parseable base ID:", dataset["Donor_Base_ID"].notna().sum())
print("Base IDs with both eyes:", len(both_eyes))
print("\nExamples of both-eye records:")
print(
    dataset[dataset["Donor_Base_ID"].isin(both_eyes.index)][
        ["Tissue_ID", "Tissue_Serial", "Eye", "Donor_Base_ID"]
    ].sort_values("Donor_Base_ID").head(20).to_string(index=False)
)

Records with a parseable base ID: 1053
Base IDs with both eyes: 0

Examples of both-eye records:
Empty DataFrame
Columns: [Tissue_ID, Tissue_Serial, Eye, Donor_Base_ID]
Index: []


## Image Dataset Validation Summary

- Clean image manifest: 1,024 tissue records.
- Images with specialist-reviewed grades: 1,009.
- Unlabelled images: 15.
- Grade distribution:
  - A+: 530
  - A: 314
  - B: 165
- Two tissue IDs require manual review because their source files contain different image contents:
  - `NEB-2024-08-16259 R`
  - `NEB-2025-09-17471 R`
- The two cases remain flagged for review rather than selecting an image arbitrarily.
- Donor-level independence has not yet been verified. The available `Tissue_Serial` values identify individual tissues, not donors.

Only labelled images will be used for supervised image classification. Data splitting must keep each tissue's images in a single split.